# TIME Benchmark Results — OpenTSLMSPFixPos2D (Gemma-3-270M)

**Evaluation:** Zero-shot, short-horizon term  
**Checkpoint:** `joint_phase2_downstream_from_joint_phase1_extend/checkpoints/best_model.pt`  
**Config:** patch_size=32, lora_r=32, lora_alpha=64, context_len=512, logit_boost=50  
**Note:** Job hit the 8-hour wall-time limit during `SG_Carpark/15T/short` (14,868 windows). Results below cover the 18 datasets that completed successfully (2 have NaN MASE due to degenerate series).

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BASE = "/home/s223540177/FoundationalTS/eval_results/time_benchmark"

rows = []
for root, dirs, files in os.walk(BASE):
    if "metrics.npz" in files and "config.json" in files:
        m   = np.load(os.path.join(root, "metrics.npz"))
        cfg = json.load(open(os.path.join(root, "config.json")))
        def safe_mean(key):
            if key in m:
                v = np.nanmean(m[key])
                return None if np.isnan(v) else float(v)
            return None
        rows.append({
            "Dataset/Term":  cfg.get("dataset_config", ""),
            "Freq":          str(cfg.get("freq", "")),
            "PredLen":       cfg.get("prediction_length"),
            "N_Series":      cfg.get("num_series"),
            "N_Windows":     cfg.get("num_windows"),
            "N_Variates":    cfg.get("num_variates"),
            "MASE":          safe_mean("MASE"),
            "CRPS":          safe_mean("CRPS"),
            "MAE":           safe_mean("MAE"),
            "MSE":           safe_mean("MSE"),
            "sMAPE":         safe_mean("sMAPE"),
        })

df = pd.DataFrame(rows).sort_values("Dataset/Term").reset_index(drop=True)
print(f"Completed dataset/term pairs: {len(df)}")
print(f"Valid MASE (non-NaN):         {df['MASE'].notna().sum()}")

## Full Results Table

In [ ]:
display_cols = ["Dataset/Term", "Freq", "PredLen", "N_Variates", "N_Windows", "MASE", "CRPS", "sMAPE"]
styled = (
    df[display_cols]
    .style
    .format({"MASE": "{:.4f}", "CRPS": "{:.4f}", "sMAPE": "{:.4f}"}, na_rep="NaN")
    .background_gradient(subset=["MASE"], cmap="RdYlGn_r", vmin=0.5, vmax=1.6)
    .background_gradient(subset=["CRPS"], cmap="Blues", vmin=0.0, vmax=1.0)
    .set_caption("Green MASE = beats seasonal naïve (< 1.0)")
)
styled

## Summary Statistics

In [ ]:
valid = df.dropna(subset=["MASE"])

print("=" * 45)
print("  TIME Benchmark Summary (short horizon)")
print("=" * 45)
print(f"  Datasets completed (total):  {len(df)}")
print(f"  Datasets with valid MASE:    {len(valid)}")
print(f"  Avg MASE:                    {valid['MASE'].mean():.4f}")
print(f"  Median MASE:                 {valid['MASE'].median():.4f}")
print(f"  Avg CRPS:                    {valid['CRPS'].mean():.4f}")
print(f"  Avg sMAPE:                   {valid['sMAPE'].mean():.4f}")
print(f"  Below MASE 1.0 (beats naïve): {(valid['MASE'] < 1.0).sum()}/{len(valid)}")
print("")
print("Top 5 (lowest MASE):")
print(valid.nsmallest(5, "MASE")[["Dataset/Term", "MASE", "CRPS"]].to_string(index=False))
print("")
print("Bottom 5 (highest MASE):")
print(valid.nlargest(5, "MASE")[["Dataset/Term", "MASE", "CRPS"]].to_string(index=False))

## MASE per Dataset

In [ ]:
plot_df = valid.sort_values("MASE")
colors  = ["#2ecc71" if v < 1.0 else "#e74c3c" for v in plot_df["MASE"]]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(plot_df["Dataset/Term"], plot_df["MASE"], color=colors, edgecolor="white", height=0.7)
ax.axvline(1.0, color="black", linewidth=1.2, linestyle="--", label="Seasonal naïve (MASE=1)")
ax.set_xlabel("MASE", fontsize=12)
ax.set_title("TIME Benchmark — MASE by Dataset (short horizon)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

for bar, val in zip(bars, plot_df["MASE"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=8)

green_patch = mpatches.Patch(color="#2ecc71", label="Beats naïve (MASE < 1)")
red_patch   = mpatches.Patch(color="#e74c3c", label="Worse than naïve (MASE ≥ 1)")
ax.legend(handles=[green_patch, red_patch,
                   plt.Line2D([0], [0], color="black", linestyle="--", label="MASE = 1.0")],
          fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(BASE, "mase_by_dataset.png"), dpi=150, bbox_inches="tight")
plt.show()

## MASE vs CRPS Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(valid["MASE"], valid["CRPS"], c=valid["MASE"], cmap="RdYlGn_r",
                vmin=0.5, vmax=1.6, s=80, edgecolors="grey", linewidths=0.5)
for _, row in valid.iterrows():
    ax.annotate(row["Dataset/Term"].split("/")[0],
                (row["MASE"], row["CRPS"]),
                fontsize=7, xytext=(4, 3), textcoords="offset points")
ax.axvline(1.0, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("MASE", fontsize=12)
ax.set_ylabel("CRPS", fontsize=12)
ax.set_title("MASE vs CRPS — TIME Benchmark (short horizon)", fontsize=13, fontweight="bold")
plt.colorbar(sc, ax=ax, label="MASE")
plt.tight_layout()
plt.savefig(os.path.join(BASE, "mase_vs_crps.png"), dpi=150, bbox_inches="tight")
plt.show()

## Incomplete / Skipped Datasets

The following datasets were **not evaluated** because the job hit the 8-hour SLURM wall time during `SG_Carpark/15T/short` (14,868 windows):

- `SG_Carpark/15T` — interrupted (14,868 windows)
- `Finland_Traffic/15T`
- `Port_Activity/D`, `Port_Activity/W`
- `ECDC_COVID/D`, `ECDC_COVID/W`
- `Global_Influenza/W`
- `Crypto/D`
- `US_Term_Structure/B`
- `Oil_Price/B`
- `Job_Claims/W`
- `Uncertainty_1M/M`
- `Housing_Inventory/M`
- `JOLTS/M`, `US_Labor/M`, `Vehicle_Supply/M`
- `Auto_Production_SF/M`, `Commodity_Production/M`, `Commodity_Import/M`
- `WUI_Global/Q`, `Global_Price/Q`
- `Vehicle_Sales/M`
- `Online_Retail_2_UCI/D`
- `Supply_Chain_Customer/D`, `Supply_Chain_Location/D`
- `azure2019_D/5T`, `azure2019_I/5T`, `azure2019_U/5T`
- `Smart_Manufacturing/H`
- `MetroPT-3/5T`

**Two datasets have NaN MASE** (`Australia_Solar/H`, `SG_Weather/D`) — likely caused by all-zero or constant ground-truth series causing degenerate MASE computation.

**Recommendation:** Re-run with a longer wall time (16–24h) or skip `SG_Carpark` using `--datasets` to exclude it.